In [1]:
# !pip install yfinance requests transformers torch xgboost scikit-learn pandas plotly

In [2]:
import yfinance as yf         # API Yahoo Finance 
import requests               # usefull for API REST HTTP of NewsAPI
import pandas as pd       
import numpy as np    
import seaborn as sns
import matplotlib.pyplot as plt       
from datetime import datetime, timedelta 
from transformers import pipeline 
import xgboost as xgb  
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc, precision_recall_curve, average_precision_score        
from sklearn.ensemble import RandomForestClassifier 
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
import plotly.figure_factory as ff # Specific Plotly module for beautiful heatmaps
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sqlite3
import os
# Create the Data directory if it doesn't exist just in case
os.makedirs("Data", exist_ok=True)
from dotenv import load_dotenv

print("All modules have been successfully imported! The environment is ready.")

All modules have been successfully imported! The environment is ready.


## I. Test Extraction Yahoo Finance

In [3]:
# Enter the company’s stock market symbol (ticker) (here Apple)
ticker = "AAPL"
action = yf.Ticker(ticker)

# Retrieve the price history for the last month (1 month)
print(f"Téléchargement des données pour {ticker}...")
df_prix = action.history(period="1mo")

# Filter to retain only the columns required for the project
colonnes_requises = ["Open", "High", "Low", "Close", "Volume"]
df_prix = df_prix[colonnes_requises]

# Display the first 5 lines to check that it works
df_prix.head()

Téléchargement des données pour AAPL...


,Open,High,Low,Close,Volume
Date,,,,,
2026-07-27 00:00:00-04:00,334.251737,339.277401,333.732166,336.619690,49604300
2026-07-28 00:00:00-04:00,339.736982,342.594533,335.310806,339.786926,51859000
2026-07-29 00:00:00-04:00,339.437272,344.273097,337.059318,337.898590,56090800
2026-07-30 00:00:00-04:00,332.812967,334.461540,329.305982,333.142670,74817800
2026-07-31 00:00:00-04:00,304.547356,310.422294,299.741503,308.643829,132489100


## II. Test Extraction NewsAPI

In [4]:
# Load the hidden variables from the .env file
load_dotenv()

# Extracting my keys from the .env
NEWS_API_KEY = os.getenv("NEWS_API_KEY")

def fetch_financial_news(query: str, days_back: int = 7) -> pd.DataFrame:
    """
    Fetches recent news articles related to a specific company or ticker.
    
    Args:
        query (str): The search term (e.g., "Apple" or "AAPL").
        days_back (int): Number of days to look back for news.
        
    Returns:
        pd.DataFrame: A dataframe containing the publication date, title, and summary of the articles.
    """
    # Calculate dates for the API request using datetime and timedelta
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days_back)
    
    # Format dates as strings (YYYY-MM-DD) required by NewsAPI
    from_param = start_date.strftime('%Y-%m-%d')
    to_param = end_date.strftime('%Y-%m-%d')
    
    # NewsAPI endpoint for searching all articles
    url = "https://newsapi.org/v2/everything"
    
    # Parameters for the API request
    params = {
        'q': query,
        'from': from_param,
        'to': to_param,
        'language': 'en',  # English news is mandatory for FinBERT compatibility
        'sortBy': 'relevancy',
        'apiKey': NEWS_API_KEY
    }
    
    # Send the HTTP GET request to the API
    response = requests.get(url, params=params)
    
    # Check if the request was successful (HTTP status code 200)
    if response.status_code == 200:
        data = response.json()
        articles = data.get('articles', [])
        
        # Extract only the relevant fields (Headline and Summary) for our NLP analysis
        extracted_data = []
        for item in articles:
            # Check if title and description exist to avoid NoneType errors
            if item.get('title') and item.get('description'):
                extracted_data.append({
                    'Date': item['publishedAt'][:10], # Keep only the YYYY-MM-DD part
                    'Headline': item['title'],
                    'Summary': item['description']
                })
            
        # Convert the list of dictionaries into a Pandas DataFrame
        df_news = pd.DataFrame(extracted_data)
        return df_news
    else:
        print(f"Error fetching news: {response.status_code} - {response.text}")
        return pd.DataFrame()

# --- Test the news extraction ---
df_apple_news = fetch_financial_news(query="Apple OR AAPL", days_back=7)

# Display the first 5 rows to visualize our text dataset
df_apple_news.head()

,Date,Headline,Summary
0,2026-08-21,"ChatGPT’s Mac App Can Now Control iMessage, Wh...",Even if Apple didn’t watch iMessage like a haw...
1,2026-08-21,Walmart is finally adding Apple Pay and Google...,Walmart will soon allow you to pay for your it...
2,2026-08-25,Bose’s smallest Bluetooth speaker is a great d...,Bose may be best known for its QuietComfort he...
3,2026-08-20,It’s Greg Brockman’s OpenAI now,OpenAI has had a hell of a year. The company s...
4,2026-08-23,That Weird ‘Other’ Section in Apple Stores Is ...,"The ""Avenues,"" as they're apparently known, re..."


## III. Analyse de Sentiment avec FinBERT

In [5]:
def analyze_financial_sentiment(df: pd.DataFrame, text_column: str = 'Headline') -> pd.DataFrame:
    """
    Applies the FinBERT NLP model to evaluate the sentiment of financial texts.
    
    Args:
        df (pd.DataFrame): The dataframe containing the news articles.
        text_column (str): The column name containing the text to analyze (Headline or Summary).
        
    Returns:
        pd.DataFrame: The original dataframe with two new columns: 'Sentiment_Label' and 'Sentiment_Score'.
    """
    # Check if the dataframe is empty to avoid errors
    if df.empty:
        print("The dataframe is empty. No sentiment analysis performed.")
        return df
        
    print("Loading FinBERT model... (This might take a minute the first time)")
    # Load the pre-trained FinBERT model specialized in finance
    sentiment_analyzer = pipeline("sentiment-analysis", model="ProsusAI/finbert")
    
    # Convert the text column to a list for the pipeline
    texts = df[text_column].tolist()
    
    print(f"Analyzing sentiment for {len(texts)} articles...")
    # Run the model on our list of texts
    results = sentiment_analyzer(texts)
    
    # Extract labels (Positive, Negative, Neutral) and confidence scores
    labels = [res['label'] for res in results]
    scores = [res['score'] for res in results]
    
    # Add the new data to our dataframe
    df_result = df.copy()
    df_result['Sentiment_Label'] = labels
    df_result['Sentiment_Score'] = scores
    
    print("Sentiment analysis completed!")
    return df_result

# --- Run the analysis on the Apple news we extracted previously ---
# Assuming your previous dataframe is named df_apple_news
df_news_with_sentiment = analyze_financial_sentiment(df_apple_news, text_column='Headline')

# Display the results
df_news_with_sentiment.head()

Loading FinBERT model... (This might take a minute the first time)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Analyzing sentiment for 89 articles...
Sentiment analysis completed!


,Date,Headline,Summary,Sentiment_Label,Sentiment_Score
0,2026-08-21,"ChatGPT’s Mac App Can Now Control iMessage, Wh...",Even if Apple didn’t watch iMessage like a haw...,neutral,0.908355
1,2026-08-21,Walmart is finally adding Apple Pay and Google...,Walmart will soon allow you to pay for your it...,neutral,0.721018
2,2026-08-25,Bose’s smallest Bluetooth speaker is a great d...,Bose may be best known for its QuietComfort he...,neutral,0.688169
3,2026-08-20,It’s Greg Brockman’s OpenAI now,OpenAI has had a hell of a year. The company s...,neutral,0.907147
4,2026-08-23,That Weird ‘Other’ Section in Apple Stores Is ...,"The ""Avenues,"" as they're apparently known, re...",neutral,0.858142


## IV. Visualisation du Sentiment

In [6]:
def plot_sentiment_distribution(df: pd.DataFrame, ticker: str):
    """
    Creates a beautiful, interactive pie chart of the sentiment distribution using Plotly.
    
    Args:
        df (pd.DataFrame): The dataframe containing the 'Sentiment_Label' column.
        ticker (str): The stock ticker to display in the title.
    """
    if 'Sentiment_Label' not in df.columns:
        print("Error: 'Sentiment_Label' column not found. Run FinBERT first.")
        return
        
    # Count the occurrences of each sentiment
    sentiment_counts = df['Sentiment_Label'].value_counts().reset_index()
    sentiment_counts.columns = ['Sentiment', 'Count']
    
    # Define a custom color palette for finance (Green=Positive, Red=Negative, Grey=Neutral)
    color_map = {
        'positive': '#2ECC71',  # Emerald Green
        'negative': '#E74C3C',  # Alizarin Red
        'neutral': '#95A5A6'    # Concrete Grey
    }
    
    # Create an interactive pie chart
    fig = px.pie(
        sentiment_counts, 
        names='Sentiment', 
        values='Count',
        title=f"<b>News Sentiment Distribution for {ticker} (Last 7 Days)</b>",
        color='Sentiment',
        color_discrete_map=color_map,
        hole=0.4 # Turns the pie chart into an elegant donut chart
    )
    
    # Upgrade the visual layout for a professional dashboard look
    fig.update_layout(
        title_font_size=20,
        font=dict(family="Arial, sans-serif", size=14),
        annotations=[dict(text='Sentiment', x=0.5, y=0.5, font_size=18, showarrow=False)]
    )
    
    # Display the interactive chart
    fig.show()

# --- Display the visual ---
plot_sentiment_distribution(df_news_with_sentiment, ticker="APPL")

## V. Feature Engineering : Indicateurs Techniques

In [7]:
def calculate_technical_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculates essential technical indicators: SMA, EMA, Volatility, and RSI.
    
    Args:
        df (pd.DataFrame): Financial dataset containing at least a 'Close' column.
        
    Returns:
        pd.DataFrame: A new dataframe enriched with technical features.
    """
    # Create a copy to avoid modifying the original data
    df_feat = df.copy()
    
    # 1. Moving Averages (20 days window)
    df_feat['SMA_20'] = df_feat['Close'].rolling(window=20).mean()
    df_feat['EMA_20'] = df_feat['Close'].ewm(span=20, adjust=False).mean()
    
    # 2. Daily Returns & Historical Volatility (14 days rolling standard deviation)
    # np.sqrt(252) is used to annualize the volatility (252 trading days in a year)
    df_feat['Daily_Return'] = df_feat['Close'].pct_change()
    df_feat['Volatility_14'] = df_feat['Daily_Return'].rolling(window=14).std() * np.sqrt(252)
    
    # 3. Relative Strength Index (RSI - 14 days)
    # Calculate the daily price differences
    delta = df_feat['Close'].diff()
    
    # Separate the gains and the losses
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    
    # Calculate the Relative Strength (RS) and RSI
    rs = gain / loss
    df_feat['RSI_14'] = 100 - (100 / (1 + rs))
    
    return df_feat

def plot_technical_dashboard(df: pd.DataFrame, ticker: str):
    """
    Generates a professional financial dashboard with Price, MAs, and RSI.
    """
    # Create a subplot grid: 2 rows, 1 column. 
    # The top chart (price) takes 70% of the space, bottom (RSI) takes 30%.
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                        row_heights=[0.7, 0.3],
                        vertical_spacing=0.05,
                        subplot_titles=(f"{ticker} - Price & Moving Averages", "Relative Strength Index (RSI)"))
                        
    # Top Chart: Price and Moving Averages
    fig.add_trace(go.Scatter(x=df.index, y=df['Close'], name='Close Price', line=dict(color='white', width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['SMA_20'], name='SMA 20', line=dict(color='#00F5FF', width=1.5)), row=1, col=1) # Cyan
    fig.add_trace(go.Scatter(x=df.index, y=df['EMA_20'], name='EMA 20', line=dict(color='#FF9900', width=1.5, dash='dot')), row=1, col=1) # Orange
    
    # Bottom Chart: RSI
    fig.add_trace(go.Scatter(x=df.index, y=df['RSI_14'], name='RSI 14', line=dict(color='#FF00FF', width=1.5)), row=2, col=1) # Magenta
    
    # Add Overbought (70) and Oversold (30) reference lines for RSI
    fig.add_hline(y=70, line_dash="dash", line_color="red", row=2, col=1, annotation_text="Overbought (70)")
    fig.add_hline(y=30, line_dash="dash", line_color="green", row=2, col=1, annotation_text="Oversold (30)")
    
    # Enhance the visual theme (Dark mode for a professional trading platform look)
    fig.update_layout(
        template='plotly_dark',
        height=700,
        hovermode='x unified', # Shows a single tooltip for all lines when hovering
        margin=dict(l=20, r=20, t=50, b=20)
    )
    
    fig.show()

# --- Execution & Test ---
# We download 6 months of data this time so the 20-day averages have enough data to compute properly
action_test = yf.Ticker("AAPL")
df_prix_6mo = action_test.history(period="6mo")[["Open", "High", "Low", "Close", "Volume"]]

# Calculate indicators
df_features = calculate_technical_indicators(df_prix_6mo)

# Display the beautiful visual
plot_technical_dashboard(df_features.tail(100), ticker="AAPL") # .tail(100) shows the last 100 days for better readability

## VI. Création de la Variable Cible (Target)

In [8]:
def create_target_variable(df: pd.DataFrame, horizon: int = 5) -> pd.DataFrame:
    """
    Creates a binary target variable for Machine Learning classification.
    
    Args:
        df (pd.DataFrame): The financial dataset containing the 'Close' price.
        horizon (int): The number of days in the future to look at (default is 5).
        
    Returns:
        pd.DataFrame: Dataset with 'Future_Close' and 'Target' columns.
                      Drops the last 'horizon' rows as their future is unknown.
    """
    df_target = df.copy()
    
    # 1. Shift the Close price backwards by 'horizon' days to align it with today's row
    df_target['Future_Close'] = df_target['Close'].shift(-horizon)
    
    # 2. Create the binary target: 1 if Future_Close > Current Close, else 0
    # We use .astype(int) to convert boolean (True/False) to 1/0
    df_target['Target'] = (df_target['Future_Close'] > df_target['Close']).astype(int)
    
    # 3. Drop rows with missing future values (the last 5 days of our dataset)
    # We can't train the model on these rows because we don't know the answer yet!
    df_target = df_target.dropna(subset=['Future_Close'])
    
    return df_target

# --- Execution ---
# Using the df_features created in the previous step
df_model_ready = create_target_variable(df_features, horizon=5)

# Display the logic to verify (Current Close vs Future Close -> Target)
df_model_ready[['Close', 'Future_Close', 'Target']].tail()

,Close,Future_Close,Target
Date,,,
2026-08-13 00:00:00-04:00,305.260010,311.299988,1
2026-08-14 00:00:00-04:00,305.929993,309.350006,1
2026-08-17 00:00:00-04:00,305.589996,310.339996,1
2026-08-18 00:00:00-04:00,310.029999,309.899994,0
2026-08-19 00:00:00-04:00,316.829987,312.894989,0


In [9]:
def plot_target_balance(df: pd.DataFrame, ticker: str):
    """
    Plots a beautiful donut chart to visualize the balance between Up and Down days.
    """
    target_counts = df['Target'].value_counts().reset_index()
    # Map 1 and 0 to readable labels
    target_counts['Target'] = target_counts['Target'].map({1: 'Up (1)', 0: 'Down (0)'})
    target_counts.columns = ['Trend', 'Count']
    
    color_map = {'Up (1)': '#2ECC71', 'Down (0)': '#E74C3C'}
    
    fig = px.pie(
        target_counts, 
        names='Trend', 
        values='Count',
        title=f"<b>Target Class Balance (5-Day Horizon) for {ticker}</b>",
        color='Trend',
        color_discrete_map=color_map,
        hole=0.5
    )
    
    fig.update_layout(template='plotly_dark', title_font_size=20)
    fig.show()

# --- Display ---
plot_target_balance(df_model_ready, ticker="AAPL")

## VII.  Création de la Base de Données SQL

In [10]:
def initialize_database(db_name: str = "finance_nlp.db"):
    """
    Connects to an SQLite database (creates it if it doesn't exist) 
    and initializes the tables according to the star schema.
    """
    # 1. Connect to the database (this creates the file in your working directory)
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    
    print(f"Connected to database: {db_name}")
    
    # 2. Create the Dimension Table (dim_assets)
    # This stores the static info about the companies we track
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS dim_assets (
        asset_id INTEGER PRIMARY KEY AUTOINCREMENT,
        ticker VARCHAR(10) NOT NULL UNIQUE,
        company_name VARCHAR(100),
        sector VARCHAR(50)
    );
    ''')
    print("Table 'dim_assets' checked/created.")
    
    # 3. Create the Fact Table for Market Data (fact_market_data)
    # Notice how we enforce uniqueness with a composite PRIMARY KEY (date, asset_id)
    # This prevents inserting the same price twice for the same day.
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS fact_market_data (
        date DATE,
        asset_id INTEGER,
        open_price DECIMAL(10,2),
        close_price DECIMAL(10,2),
        volume BIGINT,
        FOREIGN KEY (asset_id) REFERENCES dim_assets(asset_id),
        PRIMARY KEY (date, asset_id)
    );
    ''')
    print("Table 'fact_market_data' checked/created.")
    
    # 4. Create the Fact Table for News & Sentiment (fact_news_sentiment)
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS fact_news_sentiment (
        news_id INTEGER PRIMARY KEY AUTOINCREMENT,
        date DATE,
        asset_id INTEGER,
        headline TEXT,
        sentiment_score DECIMAL(5,4),
        sentiment_label VARCHAR(10),
        FOREIGN KEY (asset_id) REFERENCES dim_assets(asset_id)
    );
    ''')
    print("Table 'fact_news_sentiment' checked/created.")
    
    # 5. Commit the changes and close the connection
    conn.commit()
    conn.close()
    print("Database initialization complete.")

# --- Execution ---
initialize_database()

Connected to database: finance_nlp.db
Table 'dim_assets' checked/created.
Table 'fact_market_data' checked/created.
Table 'fact_news_sentiment' checked/created.
Database initialization complete.


## VIII. Insertion des données dans SQL (Load)

In [11]:
def insert_asset_dimension(conn: sqlite3.Connection, ticker: str, company_name: str, sector: str):
    """
    Inserts or ignores a company into the dim_assets table.
    Returns the asset_id for the given ticker.
    """
    cursor = conn.cursor()
    # INSERT OR IGNORE handles the case where the ticker already exists
    cursor.execute('''
        INSERT OR IGNORE INTO dim_assets (ticker, company_name, sector) 
        VALUES (?, ?, ?)
    ''', (ticker, company_name, sector))
    conn.commit()
    
    # Retrieve and return the asset_id
    cursor.execute('SELECT asset_id FROM dim_assets WHERE ticker = ?', (ticker,))
    result = cursor.fetchone()
    return result[0] if result else None

def insert_market_data(conn: sqlite3.Connection, df_market: pd.DataFrame, asset_id: int):
    """
    Inserts daily OHLCV data into fact_market_data.
    """
    if df_market.empty:
        return
        
    cursor = conn.cursor()
    # Convert dataframe to a list of tuples for fast bulk insertion
    records_to_insert = []
    
    for date, row in df_market.iterrows():
        # Date must be a string (YYYY-MM-DD)
        date_str = date.strftime('%Y-%m-%d')
        records_to_insert.append((
            date_str, 
            asset_id, 
            float(row['Open']), 
            float(row['Close']), 
            int(row['Volume'])
        ))
        
    # INSERT OR IGNORE prevents errors if we try to insert the same day twice
    cursor.executemany('''
        INSERT OR IGNORE INTO fact_market_data (date, asset_id, open_price, close_price, volume)
        VALUES (?, ?, ?, ?, ?)
    ''', records_to_insert)
    
    conn.commit()
    print(f"Inserted {cursor.rowcount} new price records for asset_id {asset_id}.")

# --- Execution ---
# Connect to our database
db_connection = sqlite3.connect("finance_nlp.db")

# 1. Insert Apple into the dimension table and get its ID
apple_id = insert_asset_dimension(
    conn=db_connection, 
    ticker="AAPL", 
    company_name="Apple Inc.", 
    sector="Technology"
)
print(f"Apple is registered in dim_assets with ID: {apple_id}")

# 2. Insert the historical prices we downloaded earlier (df_prix_6mo)
insert_market_data(conn=db_connection, df_market=df_prix_6mo, asset_id=apple_id)

# Always close the connection when done
db_connection.close()

Apple is registered in dim_assets with ID: 1
Inserted 2 new price records for asset_id 1.


## IX. Fin de la Phase Load : Insérer le Sentiment

In [12]:
def insert_news_sentiment(conn: sqlite3.Connection, df_news: pd.DataFrame, asset_id: int):
    """
    Inserts news articles and their FinBERT sentiment scores into fact_news_sentiment.
    """
    if df_news.empty or 'Sentiment_Label' not in df_news.columns:
        print("No sentiment data to insert.")
        return
        
    cursor = conn.cursor()
    records = []
    
    for _, row in df_news.iterrows():
        records.append((
            row['Date'],
            asset_id,
            row['Headline'],
            float(row['Sentiment_Score']),
            row['Sentiment_Label']
        ))
        
    cursor.executemany('''
        INSERT INTO fact_news_sentiment (date, asset_id, headline, sentiment_score, sentiment_label)
        VALUES (?, ?, ?, ?, ?)
    ''', records)
    
    conn.commit()
    print(f"Inserted {cursor.rowcount} news sentiment records for asset_id {asset_id}.")

# --- Execution ---
# Connect to DB
db_connection = sqlite3.connect("finance_nlp.db")

# Insert the news we analyzed earlier (df_news_with_sentiment)
insert_news_sentiment(conn=db_connection, df_news=df_news_with_sentiment, asset_id=apple_id)

db_connection.close()

df_news_with_sentiment.head()

Inserted 89 news sentiment records for asset_id 1.


,Date,Headline,Summary,Sentiment_Label,Sentiment_Score
0,2026-08-21,"ChatGPT’s Mac App Can Now Control iMessage, Wh...",Even if Apple didn’t watch iMessage like a haw...,neutral,0.908355
1,2026-08-21,Walmart is finally adding Apple Pay and Google...,Walmart will soon allow you to pay for your it...,neutral,0.721018
2,2026-08-25,Bose’s smallest Bluetooth speaker is a great d...,Bose may be best known for its QuietComfort he...,neutral,0.688169
3,2026-08-20,It’s Greg Brockman’s OpenAI now,OpenAI has had a hell of a year. The company s...,neutral,0.907147
4,2026-08-23,That Weird ‘Other’ Section in Apple Stores Is ...,"The ""Avenues,"" as they're apparently known, re...",neutral,0.858142


## X. Préparation au Machine Learning : La Fusion (Jointure SQL)

In [13]:
def fetch_training_dataset(conn: sqlite3.Connection, ticker: str) -> pd.DataFrame:
    """
    Fetches the combined dataset and aligns column names with our Python pipeline.
    """
    query = """
    WITH DailySentiment AS (
        SELECT 
            date,
            asset_id,
            AVG(CASE 
                WHEN sentiment_label = 'positive' THEN sentiment_score 
                WHEN sentiment_label = 'negative' THEN -sentiment_score 
                ELSE 0 
            END) as avg_daily_sentiment,
            COUNT(news_id) as article_count
        FROM fact_news_sentiment
        GROUP BY date, asset_id
    )
    SELECT 
        m.date,
        m.open_price AS Open,
        m.close_price AS Close,
        m.volume AS Volume,
        COALESCE(s.avg_daily_sentiment, 0) as daily_sentiment,
        COALESCE(s.article_count, 0) as news_volume
    FROM fact_market_data m
    JOIN dim_assets d ON m.asset_id = d.asset_id
    LEFT JOIN DailySentiment s ON m.date = s.date AND m.asset_id = s.asset_id
    WHERE d.ticker = ?
    ORDER BY m.date ASC
    """
    
    df_train = pd.read_sql_query(query, conn, params=(ticker,))
    df_train['date'] = pd.to_datetime(df_train['date'])
    df_train.set_index('date', inplace=True)
    return df_train


# --- Execution ---
db_connection = sqlite3.connect("finance_nlp.db")
df_ml_ready = fetch_training_dataset(conn=db_connection, ticker="AAPL")
db_connection.close()

# We recalculate technical indicators and Target on this clean, unified dataset
df_ml_ready = calculate_technical_indicators(df_ml_ready)
df_ml_ready = create_target_variable(df_ml_ready, horizon=5)

# Drop any rows with NaN values created by moving averages (first 20 days)
df_ml_ready = df_ml_ready.dropna()

print(f"Dataset ready for Machine Learning! Shape: {df_ml_ready.shape}")
df_ml_ready.tail()

Dataset ready for Machine Learning! Shape: (105, 12)


,Open,Close,Volume,daily_sentiment,news_volume,SMA_20,EMA_20,Daily_Return,Volatility_14,RSI_14,Future_Close,Target
date,,,,,,,,,,,,
2026-08-13,304.209991,305.260010,40349300,0.000000,86,319.596007,313.406970,0.009959,0.359388,29.620534,311.299988,1
2026-08-14,306.000000,305.929993,28229400,-0.966200,43,318.219887,312.694877,0.002195,0.352586,26.093530,309.350006,1
2026-08-17,306.209991,305.589996,38169300,-0.113209,414,317.183958,312.018222,-0.001111,0.346111,22.134080,311.029907,1
2026-08-18,307.579987,310.029999,53424500,-0.059186,645,316.312579,311.828867,0.014529,0.358368,28.197493,309.899994,0
2026-08-19,310.140015,316.829987,50505600,0.294137,432,315.873619,312.305164,0.021933,0.374609,37.633570,312.894989,0


## XI. Géneration d'un faux Dataset Historique

In [14]:
def load_historical_sentiment_csv(conn: sqlite3.Connection, csv_path: str, asset_id: int):
    """
    Loads a historical CSV dataset of financial news into the SQL database.
    """
    try:
        df_history = pd.read_csv(csv_path)
        
        required_cols = ['date', 'headline', 'sentiment_label', 'sentiment_score']
        if not all(col in df_history.columns for col in required_cols):
            print(f"Error: CSV must contain these columns: {required_cols}")
            return
            
        cursor = conn.cursor()
        records = []
        
        for _, row in df_history.iterrows():
            records.append((
                str(row['date'])[:10],
                asset_id,
                str(row['headline']),
                float(row['sentiment_score']),
                str(row['sentiment_label'])
            ))
            
        # Insert into fact_news_sentiment
        cursor.executemany('''
            INSERT INTO fact_news_sentiment (date, asset_id, headline, sentiment_score, sentiment_label)
            VALUES (?, ?, ?, ?, ?)
        ''', records)
        
        conn.commit()
        print(f"Successfully loaded {cursor.rowcount} historical news records for asset_id {asset_id}.")
        
    except Exception as e:
        print(f"An error occurred: {e}")

# --- Execution ---
db_connection = sqlite3.connect("finance_nlp.db")
# Load the CSV we just generated into the SQL database
load_historical_sentiment_csv(db_connection, "Data/mock_historical_news.csv", apple_id)
db_connection.close()



Successfully loaded 304 historical news records for asset_id 1.


## XII. Ingestion du Dataset Historique

In [15]:
def load_historical_sentiment_csv(conn: sqlite3.Connection, csv_path: str, asset_id: int):
    """
    Loads a historical CSV dataset of financial news into the SQL database.
    """
    try:
        df_history = pd.read_csv(csv_path)
        
        required_cols = ['date', 'headline', 'sentiment_label', 'sentiment_score']
        if not all(col in df_history.columns for col in required_cols):
            print(f"Error: CSV must contain these columns: {required_cols}")
            return
            
        cursor = conn.cursor()
        records = []
        
        for _, row in df_history.iterrows():
            records.append((
                str(row['date'])[:10],
                asset_id,
                str(row['headline']),
                float(row['sentiment_score']),
                str(row['sentiment_label'])
            ))
            
        # Insert into fact_news_sentiment
        cursor.executemany('''
            INSERT INTO fact_news_sentiment (date, asset_id, headline, sentiment_score, sentiment_label)
            VALUES (?, ?, ?, ?, ?)
        ''', records)
        
        conn.commit()
        print(f"Successfully loaded {cursor.rowcount} historical news records for asset_id {asset_id}.")
        
    except Exception as e:
        print(f"An error occurred: {e}")

# --- Execution ---
db_connection = sqlite3.connect("finance_nlp.db")
# Load the CSV we just generated into the SQL database
load_historical_sentiment_csv(db_connection, "Data/mock_historical_news.csv", apple_id)
db_connection.close()

Successfully loaded 304 historical news records for asset_id 1.


In [16]:
def fetch_training_dataset(conn, ticker: str) -> pd.DataFrame:
    """
    Fetches the combined dataset (Prices + NLP Sentiment) from SQL.
    Uses SQL aliases (AS Open, AS Close) to perfectly match our Python pipeline.
    """
    query = """
    WITH DailySentiment AS (
        SELECT 
            date,
            asset_id,
            AVG(CASE 
                WHEN sentiment_label = 'positive' THEN sentiment_score 
                WHEN sentiment_label = 'negative' THEN -sentiment_score 
                ELSE 0 
            END) as avg_daily_sentiment,
            COUNT(news_id) as article_count
        FROM fact_news_sentiment
        GROUP BY date, asset_id
    )
    SELECT 
        m.date,
        m.open_price AS Open,
        m.close_price AS Close,
        m.volume AS Volume,
        COALESCE(s.avg_daily_sentiment, 0) as daily_sentiment,
        COALESCE(s.article_count, 0) as news_volume
    FROM fact_market_data m
    JOIN dim_assets d ON m.asset_id = d.asset_id
    LEFT JOIN DailySentiment s ON m.date = s.date AND m.asset_id = s.asset_id
    WHERE d.ticker = ?
    ORDER BY m.date ASC
    """
    
    df_train = pd.read_sql_query(query, conn, params=(ticker,))
    
    # Convert dates and set as index for time series management
    df_train['date'] = pd.to_datetime(df_train['date'])
    df_train.set_index('date', inplace=True)
    
    return df_train

# --- 1. Execution of Data Pipeline ---
db_connection = sqlite3.connect("finance_nlp.db")
df_ml_ready = fetch_training_dataset(conn=db_connection, ticker="AAPL")
db_connection.close()

# Recalculate indicators and target on the 6-month joined dataset
df_ml_ready = calculate_technical_indicators(df_ml_ready)
df_ml_ready = create_target_variable(df_ml_ready, horizon=5)

# Drop rows with NaN (due to 20-day moving averages and 5-day future target)
df_ml_ready = df_ml_ready.dropna()
print(f"Final ML Dataset Shape: {df_ml_ready.shape}")

df_ml_ready.tail(20)

Final ML Dataset Shape: (105, 12)


,Open,Close,Volume,daily_sentiment,news_volume,SMA_20,EMA_20,Daily_Return,Volatility_14,RSI_14,Future_Close,Target
date,,,,,,,,,,,,
2026-07-23,321.452790,321.382843,40840800,-0.416500,90,311.223589,315.854797,-0.012980,0.238776,62.980722,333.142670,1
2026-07-24,321.512728,332.733032,47489400,0.668400,45,314.114595,317.462248,0.035317,0.273054,67.698216,308.643829,0
2026-07-27,334.251737,336.619690,49604300,0.749400,45,316.768806,319.286766,0.011681,0.269624,72.092295,303.158569,0
2026-07-28,339.736982,339.786926,51859000,0.650050,90,319.683292,321.239162,0.009409,0.269764,72.297444,309.113403,0
2026-07-29,339.437272,337.898590,56090800,0.023050,90,322.122688,322.825775,-0.005557,0.273646,68.647162,310.732025,0
2026-07-30,332.812967,333.142670,74817800,0.418350,180,324.073505,323.808336,-0.014075,0.283803,64.425689,312.140808,0
2026-07-31,304.547356,308.643829,132489100,-0.330150,90,324.087494,322.364097,-0.073539,0.434283,45.076211,313.059998,1
2026-08-03,309.313235,303.158569,75052000,-0.771575,180,323.625893,320.534999,-0.017772,0.439113,43.525022,308.260010,1
2026-08-04,302.469140,309.113403,68001000,0.839767,135,323.561948,319.447228,0.019643,0.408248,38.905182,304.910004,0


In [17]:
if 'date' not in df_ml_ready.columns:
    df_ml_ready = df_ml_ready.reset_index()

# On s'assure que c'est bien un format temporel
df_ml_ready['date'] = pd.to_datetime(df_ml_ready['date'])

df_ml_ready = df_ml_ready[df_ml_ready['date'] >= '2026-04-01']

# 1. Création du graphique (2 sous-graphiques : Sentiment en haut, Volume en bas)
fig = make_subplots(
    rows=2, cols=1, 
    shared_xaxes=True, 
    vertical_spacing=0.1,
    subplot_titles=('Average Sentiment Polarity (FinBERT)', "Financial News Volume"),
    row_heights=[0.7, 0.3]
)

# Ligne 1 : Le Sentiment NLP (En Barres de couleur)
colors = ['#00ff00' if score > 0 else '#ff0000' if score < 0 else '#ffffff' for score in df_ml_ready['daily_sentiment']]

fig.add_trace(go.Bar(
    x=df_ml_ready['date'], 
    y=df_ml_ready['daily_sentiment'], 
    name='Average Sentiment', 
    marker_color=colors,
    opacity=0.8
), row=1, col=1)

# Ligne 2 : Le Volume d'actualités (En courbe de zone)
fig.add_trace(go.Scatter(
    x=df_ml_ready['date'], 
    y=df_ml_ready['news_volume'], 
    mode='lines', 
    name='Volume', 
    line=dict(color='#0ea5e9', width=2),
    fill='tozeroy', 
    fillcolor='rgba(14, 165, 233, 0.2)' 
), row=2, col=1)

# 2. Mise en forme professionnelle (Thème Sombre)
fig.update_layout(
    template="plotly_dark",
    height=600,
    showlegend=False,
    margin=dict(l=40, r=40, t=60, b=40),
    hovermode='x unified'
)

# Ajout d'une ligne de base neutre (0) pour le sentiment
fig.add_hline(y=0, line_dash="dot", line_color="white", row=1, col=1, opacity=0.5)

# Noms des axes
fig.update_yaxes(title_text="Score (-1 à 1)", row=1, col=1)
fig.update_yaxes(title_text="Nb. Articles", row=2, col=1)

fig.show()

## XI. Application du Machine Learning

# 1) Application avec XGBoost

In [18]:
# --- Setup: Chronological Data Split ---
# Assuming df_ml_ready is your final dataframe with the 'Target' column
features = ['Open', 'Close', 'Volume', 'SMA_20', 'EMA_20', 
            'Daily_Return', 'Volatility_14', 'RSI_14', 
            'daily_sentiment', 'news_volume']

X = df_ml_ready[features]
y = df_ml_ready['Target']

split_idx = int(len(df_ml_ready) * 0.80)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print("Data successfully split: Past 80% for training, Recent 20% for testing.")

Data successfully split: Past 80% for training, Recent 20% for testing.


In [19]:
# 3. Setup TimeSeries Cross-Validation (prevents data leakage from the future)
tscv = TimeSeriesSplit(n_splits=3)

# 4. Define the hyperparameter grid to search
param_grid = {
    'n_estimators': [100, 200, 300],     # Number of trees in the forest
    'max_depth': [2, 4, 6],            # How deep the decision trees can go
    'learning_rate': [0.01, 0.05, 0.1] # How aggressively the model learns
}

# Initialize base model
base_xgb = xgb.XGBClassifier(random_state=42, eval_metric='logloss')

print("Starting Grid Search... (Testing multiple hyperparameter combinations)")

# 5. Run GridSearchCV
grid_search = GridSearchCV(
    estimator=base_xgb, 
    param_grid=param_grid, 
    cv=tscv, 
    scoring='accuracy', 
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

# 6. Extract the best model and evaluate
best_xgb_model = grid_search.best_estimator_
y_pred = best_xgb_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("\n--- OPTIMIZATION RESULTS ---")
print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Optimized Accuracy: {accuracy * 100:.2f}%")

# Generate the full classification report (Precision, Recall, F1-Score) for the synthesis report
print("\nDetailed Classification Report (F1-Score, Precision, Recall):")
print(classification_report(y_test, y_pred))

Starting Grid Search... (Testing multiple hyperparameter combinations)



--- OPTIMIZATION RESULTS ---
Best Hyperparameters: {'learning_rate': 0.01, 'max_depth': 2, 'n_estimators': 100}
Optimized Accuracy: 65.00%

Detailed Classification Report (F1-Score, Precision, Recall):
              precision    recall  f1-score   support

           0       1.00      0.42      0.59        12
           1       0.53      1.00      0.70         8

    accuracy                           0.65        20
   macro avg       0.77      0.71      0.64        20
weighted avg       0.81      0.65      0.63        20



In [20]:
# 1. Extract feature importances from our optimized model
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': best_xgb_model.feature_importances_
}).sort_values(by='Importance', ascending=True)

# 2. Plot a beautiful horizontal bar chart
fig_importance = px.bar(
    importance_df, 
    x='Importance', 
    y='Feature', 
    orientation='h',
    title="<b>Model Transparency: Feature Importance (XGBoost)</b>",
    color='Importance',
    color_continuous_scale='Mint' # Aesthetic green theme for finance
)

fig_importance.update_layout(
    template='plotly_dark', 
    title_font_size=22,
    xaxis_title="Relative Importance",
    yaxis_title="Signal"
)

fig_importance.show()

In [21]:
# 1. Calculate the confusion matrix array
cm = confusion_matrix(y_test, y_pred)

# 2. Define labels for the axes
x_labels = ['Predicted Down (0)', 'Predicted Up (1)']
y_labels = ['Actual Down (0)', 'Actual Up (1)']

# 3. Create a stylized Heatmap using Plotly Figure Factory
fig_cm = ff.create_annotated_heatmap(
    z=cm, 
    x=x_labels, 
    y=y_labels, 
    colorscale='Blues', 
    showscale=True
)

fig_cm.update_layout(
    title="<b>Confusion Matrix: Prediction vs Reality</b>",
    template='plotly_dark',
    title_font_size=22,
    xaxis=dict(title='What the Model Predicted'),
    yaxis=dict(title='What Actually Happened', autorange='reversed')
)

fig_cm.show()

In [22]:
# 1. Get prediction probabilities (confidence scores) instead of binary 0/1 labels
y_scores = best_xgb_model.predict_proba(X_test)[:, 1]

# 2. Calculate ROC metrics
fpr, tpr, roc_thresholds = roc_curve(y_test, y_scores)
roc_auc = auc(fpr, tpr)

# 3. Calculate Precision-Recall metrics
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_scores)
pr_auc = average_precision_score(y_test, y_scores)

# 4. Create a 1x2 subplot for side-by-side curves
fig_curves = make_subplots(
    rows=1, cols=2, 
    subplot_titles=(f"ROC Curve (AUC = {roc_auc:.2f})", f"Precision-Recall (AUC = {pr_auc:.2f})")
)

# --- Add ROC Curve to Subplot 1 ---
fig_curves.add_trace(go.Scatter(x=fpr, y=tpr, name='ROC', line=dict(color='#00F5FF', width=2)), row=1, col=1)
fig_curves.add_trace(go.Scatter(x=[0, 1], y=[0, 1], name='Random Guess', line=dict(color='white', dash='dash')), row=1, col=1)

# --- Add Precision-Recall Curve to Subplot 2 ---
fig_curves.add_trace(go.Scatter(x=recall, y=precision, name='PR Curve', line=dict(color='#FF00FF', width=2)), row=1, col=2)

# 5. Polish the layout
fig_curves.update_layout(
    title="<b>Advanced Model Evaluation Metrics</b>",
    template='plotly_dark',
    height=500,
    showlegend=False
)

# Update axis labels
fig_curves.update_xaxes(title_text="False Positive Rate", row=1, col=1)
fig_curves.update_yaxes(title_text="True Positive Rate", row=1, col=1)
fig_curves.update_xaxes(title_text="Recall", row=1, col=2)
fig_curves.update_yaxes(title_text="Precision", row=1, col=2)

fig_curves.show()

# 2) Application avec Random Forest

In [23]:
# 1. Définition de la grille de recherche pour Random Forest
param_grid_rf = {
    'n_estimators': [100, 300, 500],       # Nombre d'arbres dans la forêt
    'max_depth': [2, 4, 6],          # Profondeur maximale 
    'min_samples_split': [2, 4, 6, 8]        # Nombre minimum de jours pour diviser un nœud
}

# Modèle de base avec class_weight='balanced' pour compenser si les jours de hausse/baisse sont inégaux
base_rf = RandomForestClassifier(random_state=42, class_weight='balanced')

print("Starting Grid Search for Random Forest... (This may take a minute)")

# 2. Exécution de GridSearchCV (on réutilise tscv créé lors du XGBoost)
grid_search_rf = GridSearchCV(
    estimator=base_rf, 
    param_grid=param_grid_rf, 
    cv=tscv, 
    scoring='accuracy', 
    n_jobs=-1
)

grid_search_rf.fit(X_train, y_train)

# 3. Extraction du meilleur modèle et prédictions
best_rf_model = grid_search_rf.best_estimator_
y_pred_rf = best_rf_model.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_rf)

print("\n--- RANDOM FOREST OPTIMIZATION RESULTS ---")
print(f"Best Hyperparameters: {grid_search_rf.best_params_}")
print(f"Optimized Accuracy: {accuracy_rf * 100:.2f}%")

# Rapport complet exigé dans le livrable
print("\nDetailed Classification Report (F1-Score, Precision, Recall):")
print(classification_report(y_test, y_pred_rf))

Starting Grid Search for Random Forest... (This may take a minute)

--- RANDOM FOREST OPTIMIZATION RESULTS ---
Best Hyperparameters: {'max_depth': 4, 'min_samples_split': 4, 'n_estimators': 300}
Optimized Accuracy: 60.00%

Detailed Classification Report (F1-Score, Precision, Recall):
              precision    recall  f1-score   support

           0       0.83      0.42      0.56        12
           1       0.50      0.88      0.64         8

    accuracy                           0.60        20
   macro avg       0.67      0.65      0.60        20
weighted avg       0.70      0.60      0.59        20



In [24]:
# 1. Extraction des importances du modèle Random Forest
importance_df_rf = pd.DataFrame({
    'Feature': features,
    'Importance': best_rf_model.feature_importances_
}).sort_values(by='Importance', ascending=True)

# 2. Création d'un graphique à barres horizontales (Thème 'Plasma' pour différencier du XGBoost)
fig_importance_rf = px.bar(
    importance_df_rf, 
    x='Importance', 
    y='Feature', 
    orientation='h',
    title="<b>Model Transparency: Feature Importance (Random Forest)</b>",
    color='Importance',
    color_continuous_scale='Plasma' 
)

fig_importance_rf.update_layout(
    template='plotly_dark', 
    title_font_size=22,
    xaxis_title="Relative Importance",
    yaxis_title="Signal"
)

fig_importance_rf.show()

In [25]:
# 1. Calcul de la matrice de confusion pour le Random Forest
cm_rf = confusion_matrix(y_test, y_pred_rf)

# Les labels x et y ont déjà été définis dans la section XGBoost
# x_labels = ['Predicted Down (0)', 'Predicted Up (1)']
# y_labels = ['Actual Down (0)', 'Actual Up (1)']

# 2. Heatmap avec un thème 'Oranges' pour contraster avec le 'Blues' de XGBoost
fig_cm_rf = ff.create_annotated_heatmap(
    z=cm_rf, 
    x=x_labels, 
    y=y_labels, 
    colorscale='Oranges', 
    showscale=True
)

fig_cm_rf.update_layout(
    title="<b>Confusion Matrix (Random Forest)</b>",
    template='plotly_dark',
    title_font_size=22,
    xaxis=dict(title='What the Model Predicted'),
    yaxis=dict(title='What Actually Happened', autorange='reversed')
)

fig_cm_rf.show()

In [26]:
# 1. Obtenir les probabilités de prédiction (classe 1 = Hausse)
y_scores_rf = best_rf_model.predict_proba(X_test)[:, 1]

# 2. Calculer les métriques ROC
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_scores_rf)
roc_auc_rf = auc(fpr_rf, tpr_rf)

# 3. Calculer les métriques Precision-Recall
precision_rf, recall_rf, _ = precision_recall_curve(y_test, y_scores_rf)
pr_auc_rf = average_precision_score(y_test, y_scores_rf)

# 4. Créer les subplots 1x2
fig_curves_rf = make_subplots(
    rows=1, cols=2, 
    subplot_titles=(f"ROC Curve (AUC = {roc_auc_rf:.2f})", f"Precision-Recall (AUC = {pr_auc_rf:.2f})")
)

# --- Courbe ROC (Subplot 1) ---
fig_curves_rf.add_trace(go.Scatter(x=fpr_rf, y=tpr_rf, name='ROC', line=dict(color='#FFA500', width=2)), row=1, col=1) # Orange
fig_curves_rf.add_trace(go.Scatter(x=[0, 1], y=[0, 1], name='Random Guess', line=dict(color='white', dash='dash')), row=1, col=1)

# --- Courbe Precision-Recall (Subplot 2) ---
fig_curves_rf.add_trace(go.Scatter(x=recall_rf, y=precision_rf, name='PR Curve', line=dict(color='#00FF00', width=2)), row=1, col=2) # Vert vif

# 5. Mise en page
fig_curves_rf.update_layout(
    title="<b>Advanced Metrics (Random Forest)</b>",
    template='plotly_dark',
    height=500,
    showlegend=False
)

fig_curves_rf.update_xaxes(title_text="False Positive Rate", row=1, col=1)
fig_curves_rf.update_yaxes(title_text="True Positive Rate", row=1, col=1)
fig_curves_rf.update_xaxes(title_text="Recall", row=1, col=2)
fig_curves_rf.update_yaxes(title_text="Precision", row=1, col=2)

fig_curves_rf.show()

# BONUS : Test du champion_model

In [ ]:
import joblib

# 1. On "réveille" le champion en une fraction de seconde
modele_en_production = joblib.load("../Data/champion_model.pkl")

# 2. On imagine que votre pipeline ETL vient juste de récupérer les données du jour
# (Les 10 colonnes que le modèle connaît : SMA, RSI, Sentiment, etc.)
donnees_du_jour = df_ml_ready[['Open', 'Close', 'Volume', 'SMA_20', 'EMA_20', 
                                     'Daily_Return', 'Volatility_14', 'RSI_14', 
                                     'daily_sentiment', 'news_volume']]

# 3. On demande la prédiction (L'Inférence)
prediction = modele_en_production.predict(donnees_du_jour)

# 4. Interprétation du signal
if prediction[0] == 1:
    print("La tendance est qu'il faut ACHETER 📈 (Tendance haussière prévue à 5 jours)")
else:
    print("La tendance indique qu'il NE FAUT RIEN FAIRE ou VENDRE 📉 (Tendance baissière prévue)")

Signal : ACHAT 📈 (Tendance haussière prévue à 5 jours)
